In [ ]:
!pip install -q -U transformers accelerate bitsandbytes scikit-learn pandas

In [ ]:
import os
import re
import time
import warnings
import logging

import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ============================================================
# CONFIG
# ============================================================

CSV_PATH = "/kaggle/input/YOUR-DATASET/YOUR_FILE.csv"

MODEL = "md-nishat-008/TigerLLM-9B-it"

OUTPUT_DIR = "/kaggle/working/tigerllm_mcq_benchmark"

# None = full dataset
# Set to 10, 50, 100 etc. for testing
NUM_SAMPLES = None

MAX_INPUT_LENGTH = 2048
MAX_NEW_TOKENS = 32

BATCH_SIZE = 1


# ============================================================
# SETTINGS
# ============================================================

os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

warnings.filterwarnings("ignore")

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)


# ============================================================
# NORMALIZE LABEL
# ============================================================

def normalize_label(label):

    if pd.isna(label):
        return ""

    label = str(label).upper().strip()

    # Remove common formatting
    label = label.replace("OPTION", "")
    label = label.replace("ANSWER", "")
    label = label.replace(":", "")
    label = label.replace(".", "")
    label = label.replace(" ", "")
    label = label.replace(",", "")

    # Keep only A/B/C/D
    letters = [
        c for c in label
        if c in {"A", "B", "C", "D"}
    ]

    # Remove duplicate letters while keeping order
    result = ""

    for letter in letters:
        if letter not in result:
            result += letter

    return result


# ============================================================
# LOAD TIGERLLM 9B IN 8-BIT
# ============================================================

def load_local_model(model_id):

    print("Loading tokenizer...")

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"


    print("Loading TigerLLM-9B in 8-bit...")

    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True
    )


    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    model.eval()

    print("Model loaded successfully.")

    try:
        print(
            f"Model memory footprint: "
            f"{model.get_memory_footprint() / 1024**3:.2f} GB"
        )
    except Exception:
        pass

    return model, tokenizer


# ============================================================
# BUILD PROMPT
# ============================================================

def build_prompt(row):

    question = str(row["question"]).strip()

    option_a = str(row["options/A"]).strip()
    option_b = str(row["options/B"]).strip()
    option_c = str(row["options/C"]).strip()
    option_d = str(row["options/D"]).strip()


    prompt = f"""
You are answering a Bengali healthcare multiple-choice question.

Question:
{question}

Options:
A. {option_a}
B. {option_b}
C. {option_c}
D. {option_d}

Instructions:
- Select the correct answer.
- The answer may contain one option or multiple options.
- Return ONLY the option letter or letters.
- Valid answers are A, B, C, D or combinations such as AB, AC, BD.
- Do not explain.
- Do not repeat the question.
- Do not write "Answer:".

Output:
""".strip()

    return prompt


# ============================================================
# EXTRACT ANSWER FROM MODEL OUTPUT
# ============================================================

def extract_answer(text):

    if text is None:
        return ""

    text = str(text).upper().strip()

    # Take first non-empty line
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if lines:
        text = lines[0]


    # Remove common words
    text = text.replace("ANSWER", "")
    text = text.replace("OPTION", "")
    text = text.replace("CORRECT", "")
    text = text.replace(":", "")
    text = text.replace(".", "")
    text = text.replace(",", "")
    text = text.replace(" ", "")


    letters = []

    for char in text:

        if char in {"A", "B", "C", "D"}:

            if char not in letters:
                letters.append(char)


    return "".join(letters)


# ============================================================
# PREDICT ONE QUESTION
# ============================================================

@torch.inference_mode()
def predict_one(
    model,
    tokenizer,
    row
):

    prompt = build_prompt(row)


    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]


    try:

        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    except Exception:

        formatted_prompt = prompt


    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    )


    device = next(model.parameters()).device


    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    start = time.monotonic()


    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True
    )


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    latency = time.monotonic() - start


    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]


    raw_output = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


    prediction = extract_answer(
        raw_output
    )


    return prediction, raw_output, latency


# ============================================================
# CHECKPOINT
# ============================================================

def checkpoint_path(output_dir):

    return os.path.join(
        output_dir,
        "checkpoint.csv"
    )


def save_checkpoint(rows, output_dir):

    pd.DataFrame(rows).to_csv(
        checkpoint_path(output_dir),
        index=False,
        encoding="utf-8-sig"
    )


def load_checkpoint(output_dir):

    path = checkpoint_path(output_dir)

    if os.path.exists(path):

        try:
            return pd.read_csv(path)

        except Exception:
            return None

    return None


# ============================================================
# RUN MODEL ON DATASET
# ============================================================

def run_model_on_dataset(
    model,
    tokenizer,
    model_name,
    df,
    output_dir
):

    existing = load_checkpoint(
        output_dir
    )


    if existing is not None:

        rows = existing.to_dict(
            "records"
        )

    else:

        rows = []


    completed = set()

    for r in rows:

        try:
            completed.add(
                int(r["index"])
            )
        except Exception:
            pass


    total = len(df)


    for index, row in df.iterrows():

        if index in completed:
            continue


        print(
            f"\n{model_name}: "
            f"{index + 1}/{total}"
        )


        true_answer = normalize_label(
            row["answer"]
        )


        try:

            prediction, raw_output, latency = predict_one(
                model,
                tokenizer,
                row
            )


            print(
                f"True: {true_answer} | "
                f"Predicted: {prediction}"
            )


            rows.append({
                "index": index,

                "question":
                    row["question"],

                "option_A":
                    row["options/A"],

                "option_B":
                    row["options/B"],

                "option_C":
                    row["options/C"],

                "option_D":
                    row["options/D"],

                "true_answer":
                    true_answer,

                "prediction":
                    prediction,

                "raw_output":
                    raw_output,

                "correct":
                    prediction == true_answer,

                "latency":
                    latency,

                "error":
                    None
            })


        except Exception as e:

            print(
                f"Error: {e}"
            )


            rows.append({
                "index": index,

                "question":
                    row["question"],

                "option_A":
                    row["options/A"],

                "option_B":
                    row["options/B"],

                "option_C":
                    row["options/C"],

                "option_D":
                    row["options/D"],

                "true_answer":
                    true_answer,

                "prediction":
                    "",

                "raw_output":
                    "",

                "correct":
                    False,

                "latency":
                    None,

                "error":
                    str(e)
            })


        # Save after every question
        save_checkpoint(
            rows,
            output_dir
        )


        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    rows = sorted(
        rows,
        key=lambda x: int(
            x["index"]
        )
    )


    return rows


# ============================================================
# METRICS FOR SINGLE-ANSWER QUESTIONS
# ============================================================

def single_answer_metrics(rows):

    true = []
    pred = []


    for r in rows:

        t = normalize_label(
            r.get("true_answer")
        )

        p = normalize_label(
            r.get("prediction")
        )


        if len(t) == 1 and len(p) == 1:

            true.append(t)
            pred.append(p)


    if not true:

        return {
            "accuracy": None,
            "precision": None,
            "recall": None,
            "f1": None,
            "valid": 0,
            "total": 0
        }


    return {

        "accuracy":
            accuracy_score(
                true,
                pred
            ),

        "precision":
            precision_score(
                true,
                pred,
                average="weighted",
                zero_division=0
            ),

        "recall":
            recall_score(
                true,
                pred,
                average="weighted",
                zero_division=0
            ),

        "f1":
            f1_score(
                true,
                pred,
                average="weighted",
                zero_division=0
            ),

        "valid":
            len(true),

        "total":
            len([
                r for r in rows
                if len(
                    normalize_label(
                        r.get("true_answer")
                    )
                ) == 1
            ])
    }


# ============================================================
# METRICS FOR MULTI-ANSWER QUESTIONS
# ============================================================

def multi_answer_metrics(rows):

    multi_rows = []


    for r in rows:

        true_answer = normalize_label(
            r.get("true_answer")
        )

        if len(true_answer) > 1:

            multi_rows.append(r)


    if not multi_rows:

        return {
            "accuracy": None,
            "precision": None,
            "recall": None,
            "f1": None,
            "valid": 0,
            "total": 0
        }


    exact_correct = 0

    precision_values = []
    recall_values = []
    f1_values = []

    valid = 0


    for r in multi_rows:

        true_answer = normalize_label(
            r.get("true_answer")
        )

        pred_answer = normalize_label(
            r.get("prediction")
        )


        if not pred_answer:
            continue


        valid += 1


        true_set = set(
            true_answer
        )

        pred_set = set(
            pred_answer
        )


        if true_set == pred_set:

            exact_correct += 1


        intersection = len(
            true_set & pred_set
        )


        precision = (
            intersection / len(pred_set)
            if pred_set
            else 0
        )


        recall = (
            intersection / len(true_set)
            if true_set
            else 0
        )


        if precision + recall > 0:

            f1 = (
                2
                * precision
                * recall
                / (precision + recall)
            )

        else:

            f1 = 0


        precision_values.append(
            precision
        )

        recall_values.append(
            recall
        )

        f1_values.append(
            f1
        )


    return {

        "accuracy":
            exact_correct
            / len(multi_rows),

        "precision":
            sum(precision_values)
            / len(precision_values)
            if precision_values
            else 0,

        "recall":
            sum(recall_values)
            / len(recall_values)
            if recall_values
            else 0,

        "f1":
            sum(f1_values)
            / len(f1_values)
            if f1_values
            else 0,

        "valid":
            valid,

        "total":
            len(multi_rows)
    }


# ============================================================
# OVERALL METRICS
# ============================================================

def overall_metrics(rows):

    valid_rows = []

    for r in rows:

        true_answer = normalize_label(
            r.get("true_answer")
        )

        prediction = normalize_label(
            r.get("prediction")
        )

        if true_answer and prediction:

            valid_rows.append(
                (
                    true_answer,
                    prediction
                )
            )


    total = len(rows)

    valid = len(valid_rows)


    if valid == 0:

        return {
            "accuracy": 0,
            "precision": None,
            "recall": None,
            "f1": None,
            "valid": 0,
            "total": total
        }


    exact_correct = sum(
        set(t) == set(p)
        for t, p in valid_rows
    )


    precision_values = []
    recall_values = []
    f1_values = []


    for t, p in valid_rows:

        true_set = set(t)
        pred_set = set(p)


        intersection = len(
            true_set & pred_set
        )


        precision = (
            intersection
            / len(pred_set)
            if pred_set
            else 0
        )


        recall = (
            intersection
            / len(true_set)
            if true_set
            else 0
        )


        f1 = (
            2 * precision * recall
            / (precision + recall)
            if precision + recall > 0
            else 0
        )


        precision_values.append(
            precision
        )

        recall_values.append(
            recall
        )

        f1_values.append(
            f1
        )


    return {

        "accuracy":
            exact_correct / total,

        "precision":
            sum(precision_values)
            / len(precision_values),

        "recall":
            sum(recall_values)
            / len(recall_values),

        "f1":
            sum(f1_values)
            / len(f1_values),

        "valid":
            valid,

        "total":
            total
    }


# ============================================================
# COMPUTE ALL METRICS
# ============================================================

def compute_metrics(rows):

    latencies = []


    for r in rows:

        latency = r.get(
            "latency"
        )


        if (
            latency is not None
            and str(latency).lower() != "nan"
        ):

            try:

                latencies.append(
                    float(latency)
                )

            except Exception:
                pass


    return {

        "overall":
            overall_metrics(
                rows
            ),

        "single_answer":
            single_answer_metrics(
                rows
            ),

        "multi_answer":
            multi_answer_metrics(
                rows
            ),

        "avg_latency":
            sum(latencies)
            / len(latencies)
            if latencies
            else None
    }


# ============================================================
# MAIN
# ============================================================

def main():

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )


    df = pd.read_csv(
        CSV_PATH
    )


    required = {
        "question",
        "options/A",
        "options/B",
        "options/C",
        "options/D",
        "answer"
    }


    if not required.issubset(
        df.columns
    ):

        raise ValueError(
            f"CSV must contain columns: {required}"
        )


    df["answer"] = df[
        "answer"
    ].apply(
        normalize_label
    )


    # Remove rows with no answer
    df = df[
        df["answer"] != ""
    ].reset_index(
        drop=True
    )


    if NUM_SAMPLES is not None:

        df = df.head(
            NUM_SAMPLES
        ).reset_index(
            drop=True
        )


    print(
        f"\nDataset: {len(df)} rows"
    )


    print(
        f"Single-answer: "
        f"{(df['answer'].str.len() == 1).sum()}"
        f" | "
        f"Multi-answer: "
        f"{(df['answer'].str.len() > 1).sum()}"
    )


    # ========================================================
    # LOAD MODEL
    # ========================================================

    model_obj, tokenizer = load_local_model(
        MODEL
    )


    # ========================================================
    # RUN BENCHMARK
    # ========================================================

    rows = run_model_on_dataset(
        model_obj,
        tokenizer,
        MODEL,
        df,
        OUTPUT_DIR
    )


    # ========================================================
    # COMPUTE METRICS
    # ========================================================

    metrics = compute_metrics(
        rows
    )


    # ========================================================
    # SAVE PREDICTIONS
    # ========================================================

    pd.DataFrame(
        rows
    ).to_csv(

        os.path.join(
            OUTPUT_DIR,
            "predictions.csv"
        ),

        index=False,
        encoding="utf-8-sig"
    )


    # ========================================================
    # SAVE RESULTS
    # ========================================================

    results_rows = []


    for split, m in metrics.items():

        if split == "avg_latency":
            continue


        results_rows.append({

            "model":
                MODEL,

            "quantization":
                "8-bit",

            "split":
                split,

            "accuracy":
                m.get("accuracy"),

            "precision":
                m.get("precision"),

            "recall":
                m.get("recall"),

            "f1":
                m.get("f1"),

            "valid":
                m.get("valid"),

            "total":
                m.get("total"),

            "avg_latency":
                metrics["avg_latency"],
        })


    results_df = pd.DataFrame(
        results_rows
    )


    results_df.to_csv(

        os.path.join(
            OUTPUT_DIR,
            "benchmark_results.csv"
        ),

        index=False,
        encoding="utf-8-sig"
    )


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print(
        "\n── Results ──"
    )


    print(
        results_df.to_string(
            index=False
        )
    )


    print(
        "\nFiles saved to:"
    )

    print(
        OUTPUT_DIR
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()